# ResNet18 + Grad-CAM (Quỳnh) — KHUNG

**Việc của Quỳnh trước khi chạy:**
1. Cài đặt `fit / predict_proba / save / load` trong `src/banana_ripeness/models/resnet18.py` (`TODO(Quỳnh)`).
2. Cài đặt Grad-CAM trong `src/banana_ripeness/models/gradcam.py`.
3. Khai báo thí nghiệm trong `configs/experiments.yaml` (mục `quynh`; đang để 3 seed để tính std).
4. Push code lên GitHub, rồi chạy notebook này trên Kaggle (**Accelerator: GPU**, Internet On).

Khung đã có: đọc `stats.json` (`train_mean_std()`, `train_class_weight()`), `load_image(...)`, chia dữ liệu,
độ đo, bootstrap std, std giữa các seed, lưu kết quả cùng định dạng để notebook `compare_results` gộp với SVM / HOG.

In [ ]:
# === SETUP: chạy cell này đầu tiên (Kaggle / Colab / máy) ===
# Lấy code .py theo thứ tự:
#   1. chạy trên máy trong thư mục repo
#   2. code .py đã UPLOAD lên Kaggle (Add Input -> dataset chứa thư mục src/ + configs/) hoặc Colab (/content)
#   3. git clone từ GitHub (Kaggle: bật Settings -> Internet = On)
import os, sys, shutil, subprocess
from pathlib import Path

CODE_SOURCE = "auto"       # "auto" | "upload" | "git"
REPO_URL = "https://github.com/LuanLuKim/cs231-banana-ripeness.git"
BRANCH = "main"
OWNER = "quynh"          # tên người chạy, ghi vào bảng kết quả chung

def find_uploaded_code(roots=("/kaggle/input", "/content", "/content/drive/MyDrive"), max_depth=5):
    """Thư mục gốc của code đã upload (chứa src/banana_ripeness/__init__.py hoặc banana_ripeness/__init__.py)."""
    skip = {"images", "masks", "results", "cache", "reports", ".git", "drive", "sample_data", "__pycache__"}
    for base in map(Path, roots):
        if not base.exists():
            continue
        for dirpath, dirnames, filenames in os.walk(base):
            d = Path(dirpath)
            if d.name == "banana_ripeness" and "__init__.py" in filenames:
                return d.parent.parent if d.parent.name == "src" else d.parent
            depth = len(d.parts) - len(base.parts)
            dirnames[:] = [] if depth >= max_depth else sorted(x for x in dirnames if x not in skip)
    return None

local = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "banana_ripeness").exists()), None)
uploaded = find_uploaded_code() if CODE_SOURCE in ("auto", "upload") and local is None else None
if local is not None and CODE_SOURCE == "auto":
    CODE_DIR, source = local, "máy"
elif uploaded is not None:
    # chép sang /tmp: thư mục /kaggle/input chỉ đọc, và để output notebook không chứa code
    CODE_DIR = Path("/tmp/cs231-code")
    shutil.rmtree(CODE_DIR, ignore_errors=True)
    shutil.copytree(uploaded, CODE_DIR, ignore=shutil.ignore_patterns("__pycache__", ".git", "*.ipynb"))
    source = f"upload ({uploaded})"
elif CODE_SOURCE == "upload":
    raise FileNotFoundError("Không thấy code đã upload: Add Input dataset chứa src/banana_ripeness/ và configs/")
else:
    CODE_DIR = Path("/tmp/cs231-banana-ripeness")
    if (CODE_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(CODE_DIR), "pull", "-q", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO_URL, str(CODE_DIR)], check=True)
    commit = subprocess.run(["git", "-C", str(CODE_DIR), "rev-parse", "--short", "HEAD"],
                            capture_output=True, text=True).stdout.strip()
    source = f"git {commit}"
SRC = CODE_DIR / "src" if (CODE_DIR / "src" / "banana_ripeness").exists() else CODE_DIR
sys.path.insert(0, str(SRC))
for m in [m for m in sys.modules if m == "banana_ripeness" or m.startswith("banana_ripeness.")]:
    del sys.modules[m]        # chạy lại cell sau khi đổi code -> nạp bản mới
os.environ["BANANA_OWNER"] = OWNER
print("Code:", CODE_DIR, "| nguồn:", source)
if not (CODE_DIR / "configs").exists():
    print("!! Thiếu thư mục configs/ (cần configs/kaggle.yaml, configs/experiments.yaml) — upload cả thư mục configs")

%load_ext autoreload
%autoreload 2

In [ ]:
from banana_ripeness.data import find_data_root, load_manifest, get_split
import pandas as pd
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)

# Tự tìm thư mục có manifest.csv trong /kaggle/input (Kaggle Dataset output preprocessing đã Add Input).
# Nếu có nhiều bản, chỉ rõ: DATA_ROOT = find_data_root("/kaggle/input/<ten-dataset>")
DATA_ROOT = find_data_root()
df = load_manifest(DATA_ROOT)
EXP = CODE_DIR / "configs" / "experiments.yaml"
print("Dữ liệu:", DATA_ROOT, "|", len(df), "ảnh")
pd.crosstab([df.split_mixed], df.label).loc[["train", "val", "test"]]

In [ ]:
from banana_ripeness.experiment import load_experiments, run_grid, load_results, compare_table
res = run_grid(load_experiments(EXP, "quynh"), df=df, data_root=DATA_ROOT, owner=OWNER)
res = load_results(); res = res[res.owner == OWNER]
display(compare_table(res, split="test", scheme="mixed"))        # std giữa các seed nếu > 1 seed
display(compare_table(res, split="external", scheme="external"))

In [ ]:
from banana_ripeness.experiment import best_run
from banana_ripeness.models import load_model
best = best_run(res, scheme="mixed", by_split="val")
best.plot_confusion("test");
model = load_model(best.run_dir)

In [ ]:
# Grad-CAM (TODO Quỳnh: cài đặt gradcam_grid)
from banana_ripeness.models.gradcam import gradcam_grid
test = get_split(df, "mixed", "test")
gradcam_grid(model, test.groupby("label").sample(2, random_state=42));